# 📊 Datos, Limpieza, Features y Métricas Base

En este notebook vas a aprender a **trabajar con datos reales**: cargarlos, explorarlos, limpiarlos,
partirlos correctamente y medir qué tan bien funciona un modelo.

### ¿Qué vamos a cubrir?

| Sección | Concepto |
|---|---|
| 1 | Instalación y verificación de herramientas |
| 2 | Cargar y explorar un dataset |
| 3 | Tipos de datos y estadísticas descriptivas |
| 4 | Limpieza: valores faltantes y outliers |
| 5 | Features: ¿qué le damos al modelo? |
| 6 | Partición train / validation / test |
| 7 | Métricas de evaluación |
| 8 | Sesgo y varianza: la intuición |
| 9 | Ejercicio evaluable |

_
> **Prerequisito:** Haber completado NB1 (conceptos básicos de IA). No se necesita experiencia con pandas — todo se explica paso a paso.

---
## 1. Instalación y verificación

Necesitamos tres librerías:
- **pandas** → Manipulación de datos tabulares
- **matplotlib** → Gráficos
- **scikit-learn** → Partición de datos y métricas

```bash
pip install pandas matplotlib scikit-learn
```

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print(f"✅ pandas       : {pd.__version__}")
print(f"✅ numpy        : {np.__version__}")
print(f"✅ matplotlib   : {plt.matplotlib.__version__}")

# Configuración visual para todo el notebook
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("\n📍 Entorno listo para trabajar.")

✅ pandas       : 3.0.1
✅ numpy        : 2.4.3
✅ matplotlib   : 3.10.8

📍 Entorno listo para trabajar.


---
## 2. Cargar y explorar un dataset

Vamos a trabajar con un dataset **sintético** que simula datos de estudiantes.
Así no dependemos de descargas externas y podemos controlar exactamente qué problemas tienen los datos.

### ¿Qué contiene el dataset?

| Columna | Descripción | Tipo |
|---|---|---|
| `horas_estudio` | Horas de estudio semanales | Numérico |
| `asistencia_pct` | % de asistencia a clases | Numérico |
| `trabajos_entregados` | Cantidad de trabajos entregados (de 10) | Numérico |
| `horas_sueno` | Horas promedio de sueño por noche | Numérico |
| `metodo_estudio` | Método principal de estudio | Categórico |
| `nota_final` | Nota final del curso (0-10) | Numérico (target) |

_
> 💡 En ML, a las columnas de entrada les llamamos **features** y a la columna que queremos predecir, **target**.

In [2]:
# Generar un dataset sintético reproducible
np.random.seed(42)  # semilla para reproducibilidad

n = 200  # cantidad de estudiantes

horas_estudio       = np.random.uniform(0.5, 15, n)
asistencia_pct      = np.clip(np.random.normal(75, 15, n), 10, 100)
trabajos_entregados = np.random.randint(0, 11, n)
horas_sueno         = np.clip(np.random.normal(7, 1.5, n), 3, 12)
metodo_estudio      = np.random.choice(
    ["Lectura", "Videos", "Práctica", "Grupo", "Ninguno"], n,
    p=[0.25, 0.2, 0.3, 0.15, 0.1]
)

# Nota final = combinación con ruido (simula relación real)
nota_final = (
    0.3 * horas_estudio
    + 0.02 * asistencia_pct
    + 0.25 * trabajos_entregados
    + 0.15 * horas_sueno
    + np.random.normal(0, 0.8, n)  # ruido
)
nota_final = np.clip(nota_final, 0, 10)

# Crear DataFrame
df = pd.DataFrame({
    "horas_estudio":       np.round(horas_estudio, 1),
    "asistencia_pct":      np.round(asistencia_pct, 1),
    "trabajos_entregados": trabajos_entregados,
    "horas_sueno":         np.round(horas_sueno, 1),
    "metodo_estudio":      metodo_estudio,
    "nota_final":          np.round(nota_final, 2),
})

# Introducir datos faltantes a propósito (realismo)
indices_nulos = np.random.choice(n, 15, replace=False)
df.loc[indices_nulos[:8], "asistencia_pct"] = np.nan
df.loc[indices_nulos[8:12], "horas_sueno"]  = np.nan
df.loc[indices_nulos[12:], "metodo_estudio"] = np.nan

# Introducir outliers a propósito
df.loc[5, "horas_estudio"]  = 50     # imposible: ¿50 horas semanales?
df.loc[42, "asistencia_pct"] = 150    # error: >100%

print(f"📊 Dataset creado: {df.shape[0]} filas × {df.shape[1]} columnas\n")
print(df.head(10).to_string())

📊 Dataset creado: 200 filas × 6 columnas

   horas_estudio  asistencia_pct  trabajos_entregados  horas_sueno metodo_estudio  nota_final
0            5.9            64.8                   10          5.8        Ninguno        7.14
1           14.3            78.5                    9          6.2        Lectura        9.26
2           11.1            79.4                    9          4.8         Videos        6.82
3            9.2            64.3                    2          6.7        Lectura        6.02
4            2.8           100.0                    6          7.1        Ninguno        5.96
5           50.0            82.1                    2          3.6       Práctica        3.54
6            1.3            57.1                    1          6.8       Práctica        3.80
7           13.1            84.8                    9          7.1        Lectura        8.38
8            9.2            60.4                    3          7.8       Práctica        5.48
9           10.8  

---
## 3. Tipos de datos y estadísticas descriptivas

Antes de hacer cualquier cosa con los datos, **siempre** hay que explorarlos primero.

### Las 3 preguntas obligatorias

1. **¿Qué forma tienen?** → Filas, columnas
2. **¿Qué tipo es cada columna?** → Numérica, categórica, fecha
3. **¿Hay datos faltantes?** → NaN, vacíos, errores

> 💡 **Regla de oro:** nunca entrenes un modelo sin haber respondido estas tres preguntas.

In [ ]:
# Pregunta 1: ¿Qué forma tienen los datos?
print(f"📐 Forma del dataset: {df.shape[0]} filas × {df.shape[1]} columnas\n")

# Pregunta 2: Tipos de cada columna
print("📋 Tipos de datos:")
print(df.dtypes.to_string())

print(f"\n{'─' * 50}")

# Pregunta 3: ¿Hay valores faltantes?
nulos = df.isnull().sum()
nulos_total = nulos.sum()
print(f"\n🔍 Valores faltantes (NaN): {nulos_total} en total")
if nulos_total > 0:
    print()
    for col, cnt in nulos.items():
        if cnt > 0:
            emoji = "⚠️" if cnt > 5 else "🟡"
            print(f"   {emoji} {col}: {cnt} faltantes ({cnt/len(df)*100:.1f}%)")

print(f"\n{'─' * 50}")

# Estadísticas descriptivas de las columnas numéricas
print("\n📊 Estadísticas descriptivas:\n")
print(df.describe().round(2).to_string())

---
## 4. Limpieza: valores faltantes y outliers

Los datos reales **siempre** tienen problemas. Los más comunes:

| Problema | Qué es | Ejemplo |
|---|---|---|
| **Valores faltantes** | Celdas vacías o NaN | Un estudiante no completó la encuesta |
| **Outliers** | Valores extremos o imposibles | Horas de estudio = 50 (¿por semana?) |
| **Errores de rango** | Valores fuera del dominio lógico | Asistencia = 150% |

### Estrategias para valores faltantes

| Estrategia | Cuándo usarla |
|---|---|
| **Eliminar la fila** | Pocos faltantes, muchos datos |
| **Rellenar con la media/mediana** | Columna numérica, distribución simétrica |
| **Rellenar con la moda** | Columna categórica |
| **Rellenar con un valor especial** | Cuando "faltante" tiene significado propio |

> 💡 **Nunca ignores los NaN.** Muchos modelos de ML fallan silenciosamente con datos faltantes.

In [ ]:
# Hacemos una copia para no modificar el original
df_limpio = df.copy()

print("🧹 PASO 1: Corregir outliers y errores de rango\n")

# Horas de estudio: máximo razonable = 20 horas semanales
outliers_horas = df_limpio["horas_estudio"] > 20
print(f"   ⚠️  Outliers en horas_estudio (>20): {outliers_horas.sum()} encontrados")
df_limpio.loc[outliers_horas, "horas_estudio"] = df_limpio["horas_estudio"].median()
print(f"   ✅ Reemplazados por la mediana: {df_limpio['horas_estudio'].median():.1f}")

# Asistencia: debe estar entre 0 y 100
outliers_asist = df_limpio["asistencia_pct"] > 100
print(f"\n   ⚠️  Errores en asistencia_pct (>100%): {outliers_asist.sum()} encontrados")
df_limpio.loc[outliers_asist, "asistencia_pct"] = 100.0
print(f"   ✅ Recortados a 100%")

print(f"\n{'─' * 50}")
print("\n🧹 PASO 2: Tratar valores faltantes\n")

# Numéricos: rellenar con la mediana (robusta a outliers)
for col in ["asistencia_pct", "horas_sueno"]:
    n_faltantes = df_limpio[col].isnull().sum()
    if n_faltantes > 0:
        valor_relleno = df_limpio[col].median()
        df_limpio[col] = df_limpio[col].fillna(valor_relleno)
        print(f"   ✅ {col}: {n_faltantes} NaN → rellenados con mediana ({valor_relleno:.1f})")

# Categórico: rellenar con la moda (valor más frecuente)
n_faltantes_metodo = df_limpio["metodo_estudio"].isnull().sum()
if n_faltantes_metodo > 0:
    moda = df_limpio["metodo_estudio"].mode()[0]
    df_limpio["metodo_estudio"] = df_limpio["metodo_estudio"].fillna(moda)
    print(f"   ✅ metodo_estudio: {n_faltantes_metodo} NaN → rellenados con moda ('{moda}')")

# Verificación final
print(f"\n{'─' * 50}")
print(f"\n🔍 Valores faltantes después de limpieza: {df_limpio.isnull().sum().sum()}")
print(f"📊 Dataset limpio: {df_limpio.shape[0]} filas × {df_limpio.shape[1]} columnas")

In [ ]:
# Visualización antes vs. después de la limpieza
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Antes
axes[0].hist(df["horas_estudio"].dropna(), bins=20, color="#ff6b6b", alpha=0.7, edgecolor="white")
axes[0].set_title("ANTES de limpiar", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Horas de estudio")
axes[0].axvline(50, color="red", linestyle="--", label="Outlier (50h)")
axes[0].legend()

# Después
axes[1].hist(df_limpio["horas_estudio"], bins=20, color="#6bcb77", alpha=0.7, edgecolor="white")
axes[1].set_title("DESPUÉS de limpiar", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Horas de estudio")

plt.suptitle("Efecto de la limpieza de outliers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("📌 El outlier de 50 horas desaparece después de la limpieza.")

---
## 5. Features: ¿qué le damos al modelo?

Las **features** (características) son las columnas que el modelo usa como entrada para hacer predicciones.

### Tipos de features

| Tipo | Ejemplo | Tratamiento |
|---|---|---|
| **Numérica continua** | horas_estudio, asistencia_pct | Usar directo (opcionalmente escalar) |
| **Numérica discreta** | trabajos_entregados | Usar directo |
| **Categórica** | metodo_estudio | Convertir a números (encoding) |

### ¿Qué es el encoding?

Los modelos de ML no entienden texto — necesitan números. El **encoding** transforma categorías en valores numéricos.

```
"Lectura"   → 0
"Videos"    → 1
"Práctica"  → 2
"Grupo"     → 3
"Ninguno"   → 4
```

> 💡 **Cuidado:** este encoding simple (label encoding) implica un orden que no existe. Para modelos sensibles al orden, se usa **one-hot encoding**.

In [ ]:
# Separar features y target
print("🎯 Separando features (X) y target (y)\n")

# El target es lo que queremos predecir: la nota final
target = "nota_final"

# Las features son todo lo demás
features_numericas  = ["horas_estudio", "asistencia_pct", "trabajos_entregados", "horas_sueno"]
features_categoricas = ["metodo_estudio"]

print(f"📊 Features numéricas  : {features_numericas}")
print(f"📊 Features categóricas: {features_categoricas}")
print(f"🎯 Target             : {target}")

# ── Encoding de la variable categórica ──────────────────────
print(f"\n{'─' * 50}")
print("\n🔄 Encoding de 'metodo_estudio' (one-hot):\n")

# One-hot encoding: cada categoría se convierte en una columna binaria
df_encoded = pd.get_dummies(df_limpio, columns=["metodo_estudio"], prefix="metodo")

# Mostrar las nuevas columnas
columnas_nuevas = [c for c in df_encoded.columns if c.startswith("metodo_")]
print(f"   Columnas originales: ['metodo_estudio']")
print(f"   Columnas nuevas    : {columnas_nuevas}\n")
print(df_encoded[columnas_nuevas].head(5).to_string())

# Preparar X e y finales
feature_cols = features_numericas + columnas_nuevas
X = df_encoded[feature_cols]
y = df_encoded[target]

print(f"\n{'─' * 50}")
print(f"\n✅ X (features): {X.shape[0]} filas × {X.shape[1]} columnas")
print(f"✅ y (target)  : {y.shape[0]} valores")

---
## 6. Partición train / validation / test

Nunca evaluamos un modelo con los mismos datos que usamos para entrenarlo.
Eso sería como hacer un examen con las respuestas al lado.

### Los 3 conjuntos

| Conjunto | Para qué se usa | % típico |
|---|---|---|
| **Train** | El modelo aprende de estos datos | 70% |
| **Validation** | Ajustar hiperparámetros, evitar overfitting | 15% |
| **Test** | Evaluación final — **se usa UNA sola vez** | 15% |

### ¿Por qué es tan importante?

```
Si evalúo con datos de train → el modelo "se sabe las respuestas" → métricas infladas
Si evalúo con datos nuevos   → evaluación honesta del modelo
```

> 💡 **Analogía:** Train es estudiar, Validation es hacer simulacros de examen, Test es el examen final real.

In [ ]:
from sklearn.model_selection import train_test_split

# Paso 1: separar test (15% del total)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size    = 0.15,    # 15% para test
    random_state = 42       # semilla para reproducibilidad
)

# Paso 2: del 85% restante, separar validation (~17.6% de temp ≈ 15% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size    = 0.176,   # ~15% del total original
    random_state = 42
)

print("📊 Partición de datos\n")
print(f"   {'Conjunto':<15} {'Filas':<10} {'% del total'}")
print(f"   {'─' * 40}")

total = len(X)
for nombre, datos_x in [("Train", X_train), ("Validation", X_val), ("Test", X_test)]:
    pct = len(datos_x) / total * 100
    print(f"   {nombre:<15} {len(datos_x):<10} {pct:.1f}%")

print(f"   {'─' * 40}")
print(f"   {'TOTAL':<15} {total:<10} 100.0%")

print("\n💡 El set de TEST no se toca hasta la evaluación final.")

In [ ]:
# Visualización de la partición
fig, ax = plt.subplots(figsize=(10, 3))

tamaños = [len(X_train), len(X_val), len(X_test)]
etiquetas = [
    f"Train\n({tamaños[0]} filas)",
    f"Validation\n({tamaños[1]} filas)",
    f"Test\n({tamaños[2]} filas)"
]
colores = ["#4d96ff", "#ffd93d", "#ff6b6b"]

inicio = 0
for tam, etiq, color in zip(tamaños, etiquetas, colores):
    ax.barh(0, tam, left=inicio, color=color, edgecolor="white", height=0.6)
    ax.text(inicio + tam/2, 0, etiq, ha="center", va="center",
            fontsize=11, fontweight="bold")
    inicio += tam

ax.set_xlim(0, total)
ax.set_ylim(-0.5, 0.5)
ax.axis("off")
ax.set_title("Partición del dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Métricas de evaluación

¿Cómo sabemos si un modelo es bueno? Necesitamos **métricas numéricas** que midan el error.

### Métricas para regresión (predecir un número)

| Métrica | Fórmula intuitiva | Interpretación |
|---|---|---|
| **MAE** (Error Absoluto Medio) | Promedio de `|real - predicho|` | "En promedio me equivoco por X unidades" |
| **MSE** (Error Cuadrático Medio) | Promedio de `(real - predicho)²` | Penaliza errores grandes más que MAE |
| **RMSE** (Raíz de MSE) | `√MSE` | Misma unidad que el target |
| **R²** (Coeficiente de determinación) | 0 a 1 | "El modelo explica X% de la variación" |

### Métricas para clasificación (predecir una categoría)

| Métrica | Qué mide |
|---|---|
| **Accuracy** | % de predicciones correctas |
| **Precision** | De las que predije positivo, ¿cuántas realmente lo eran? |
| **Recall** | De las que realmente eran positivo, ¿cuántas encontré? |
| **F1-Score** | Balance entre Precision y Recall |

> 💡 **Para nuestro caso** (predecir nota = número continuo) usaremos métricas de **regresión**.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Entrenar un modelo simple: regresión lineal
modelo = LinearRegression()
modelo.fit(X_train, y_train)  # aprende de los datos de train

# Predecir en cada conjunto
y_pred_train = modelo.predict(X_train)
y_pred_val   = modelo.predict(X_val)

print("📊 Métricas de evaluación\n")
print(f"{'Métrica':<10} {'Train':<12} {'Validation':<12} {'Interpretación'}")
print("─" * 65)

# MAE
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_val   = mean_absolute_error(y_val, y_pred_val)
print(f"{'MAE':<10} {mae_train:<12.3f} {mae_val:<12.3f} Error promedio en unidades de nota")

# RMSE
rmse_train = mean_squared_error(y_train, y_pred_train, squared=False)
rmse_val   = mean_squared_error(y_val, y_pred_val, squared=False)
print(f"{'RMSE':<10} {rmse_train:<12.3f} {rmse_val:<12.3f} Penaliza errores grandes")

# R²
r2_train = r2_score(y_train, y_pred_train)
r2_val   = r2_score(y_val, y_pred_val)
print(f"{'R²':<10} {r2_train:<12.3f} {r2_val:<12.3f} % de varianza explicada (1 = perfecto)")

print(f"\n💡 Si las métricas de train son MUCHO mejores que validation → posible overfitting.")

diferencia_r2 = abs(r2_train - r2_val)
if diferencia_r2 < 0.05:
    print(f"✅ Diferencia de R² = {diferencia_r2:.3f} → el modelo generaliza bien.")
else:
    print(f"⚠️  Diferencia de R² = {diferencia_r2:.3f} → posible overfitting, investigar.")

In [ ]:
# Gráfico: valores reales vs. predichos
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_real, y_pred, titulo, color in [
    (axes[0], y_train, y_pred_train, "Train", "#4d96ff"),
    (axes[1], y_val,   y_pred_val,   "Validation", "#ffd93d"),
]:
    ax.scatter(y_real, y_pred, alpha=0.5, color=color, edgecolor="gray", s=40)
    ax.plot([0, 10], [0, 10], "--", color="red", linewidth=1.5, label="Predicción perfecta")
    ax.set_xlabel("Nota real", fontsize=11)
    ax.set_ylabel("Nota predicha", fontsize=11)
    ax.set_title(f"{titulo}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_xlim(0, 10.5)
    ax.set_ylim(0, 10.5)
    ax.set_aspect("equal")

plt.suptitle("Real vs. Predicho — ¿qué tan bueno es el modelo?", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("📌 Cuanto más cerca de la línea roja, mejor es la predicción.")

---
## 8. Sesgo y varianza: la intuición

Dos conceptos fundamentales para entender **por qué un modelo falla**.

### Definiciones simples

| Concepto | Significado | Señal |
|---|---|---|
| **Sesgo alto** (underfitting) | El modelo es demasiado simple para capturar el patrón | Error alto en train Y validation |
| **Varianza alta** (overfitting) | El modelo memoriza los datos de entrenamiento | Error bajo en train, alto en validation |

### La analogía del tiro al blanco

```
                 Bajo sesgo          Alto sesgo
              ┌──────────────┐  ┌──────────────┐
  Baja        │    ·         │  │         · ·  │
  varianza    │     ·  ·    │  │        · ·   │
              │    · 🎯      │  │       🎯     │
              └──────────────┘  └──────────────┘
              Ideal ✅           Consistente pero mal ❌

              ┌──────────────┐  ┌──────────────┐
  Alta        │  ·     ·     │  │ ·       ·    │
  varianza    │     🎯    ·  │  │    🎯  ·     │
              │  ·      ·    │  │  ·         · │
              └──────────────┘  └──────────────┘
              Inestable ⚠️       Lo peor ❌❌
```

> 💡 **El balance sesgo-varianza** es la tensión central en ML: un modelo más complejo reduce sesgo pero puede aumentar varianza.

In [ ]:
# Demostración interactiva: underfitting vs. overfitting
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Datos simples para la demo
np.random.seed(42)
x_demo = np.sort(np.random.uniform(0, 10, 30))
y_demo = 0.5 * x_demo**2 - 3 * x_demo + 10 + np.random.normal(0, 3, 30)  # parábola con ruido

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x_curva = np.linspace(0, 10, 100).reshape(-1, 1)

configuraciones = [
    {"grado": 1,  "titulo": "Underfitting\n(Sesgo alto)",   "color": "#ff6b6b"},
    {"grado": 2,  "titulo": "Ajuste correcto\n(Balance)",   "color": "#6bcb77"},
    {"grado": 15, "titulo": "Overfitting\n(Varianza alta)", "color": "#ffd93d"},
]

for ax, cfg in zip(axes, configuraciones):
    # Entrenar modelo polinomial de grado N
    modelo_poly = make_pipeline(
        PolynomialFeatures(cfg["grado"]),
        LinearRegression()
    )
    modelo_poly.fit(x_demo.reshape(-1, 1), y_demo)
    y_curva = modelo_poly.predict(x_curva)

    ax.scatter(x_demo, y_demo, color="#333", s=30, alpha=0.7, label="Datos")
    ax.plot(x_curva, y_curva, color=cfg["color"], linewidth=2.5, label=f"Grado {cfg['grado']}")
    ax.set_title(cfg["titulo"], fontsize=12, fontweight="bold")
    ax.set_ylim(-10, 40)
    ax.legend(fontsize=9)

plt.suptitle("Sesgo vs. Varianza: el dilema central de ML", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("📌 Izquierda: modelo demasiado simple (no captura la curva)")
print("📌 Centro:    modelo justo (captura el patrón sin memorizar ruido)")
print("📌 Derecha:   modelo demasiado complejo (memoriza cada punto)")

---
## 9. Ejercicio evaluable: análisis completo de un dataset

### Tu entregable

Completá las celdas a continuación con tu propio análisis. Usá el mismo dataset que generamos (`df`) o podés crear uno similar.

### Criterio de aprobación

- ✅ Identificar correctamente los tipos de features
- ✅ Aplicar al menos una técnica de limpieza
- ✅ Particionar en train/val/test correctamente
- ✅ Calcular y reportar al menos 2 métricas
- ✅ Justificar la elección de métricas (1-2 oraciones)

> 💡 **Tip:** no busques un modelo perfecto — lo importante es demostrar que entendés el proceso completo.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Parte 1: Exploración
# ══════════════════════════════════════════════════════════════

# 1. ¿Cuántas filas y columnas tiene el dataset limpio?
n_filas   = ...   # TODO: completar
n_columnas = ...  # TODO: completar

# 2. ¿Cuántas features numéricas y cuántas categóricas hay?
n_numericas   = ...  # TODO: completar
n_categoricas = ...  # TODO: completar

# 3. ¿Cuál es la correlación más fuerte con la nota final?
# Tip: usá df_limpio.corr(numeric_only=True)["nota_final"].sort_values(ascending=False)
feature_mas_correlacionada = ""  # TODO: completar con el nombre de la columna

print(f"Filas: {n_filas}, Columnas: {n_columnas}")
print(f"Features numéricas: {n_numericas}, categóricas: {n_categoricas}")
print(f"Feature más correlacionada: {feature_mas_correlacionada}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Parte 2: Modelado y métricas
# ══════════════════════════════════════════════════════════════

# 1. Entrená un modelo de regresión lineal con los datos de train
# TODO: completar

# 2. Calculá MAE y R² en train y validation
# TODO: completar

# 3. ¿Hay señales de overfitting? Justificá en 1-2 oraciones.
justificacion_overfitting = ""  # TODO: completar

# 4. ¿Qué métrica elegirías como principal y por qué?
metrica_elegida    = ""  # TODO: "MAE" o "R²" u otra
justificacion_metrica = ""  # TODO: 1-2 oraciones

print(f"Justificación overfitting: {justificacion_overfitting}")
print(f"Métrica elegida: {metrica_elegida}")
print(f"Justificación: {justificacion_metrica}")

---
## Resumen del notebook

| Concepto | Lo que aprendiste |
|---|---|
| Exploración | Las 3 preguntas obligatorias antes de modelar |
| Limpieza | Tratar NaN, outliers y errores de rango |
| Features | Numéricas, categóricas y encoding |
| Partición | Train / Validation / Test y por qué importa |
| Métricas | MAE, RMSE, R² para regresión |
| Sesgo/Varianza | Underfitting vs. overfitting — la tensión central de ML |

### Próximo notebook

En **NB3** vamos a construir tu primera **red neuronal** con PyTorch: capas, pérdida, optimización y entrenamiento visual.

> 📌 **Recordá:** datos limpios + partición correcta + métricas honestas = la base de cualquier proyecto de IA serio.